In [ ]:
import numpy as np
import pandas as pd
from scipy.special import gammaln
from scipy.stats import norm
import math

n = 20  # Number of samples or lineages at start 
# N_e1 = 1500  # Effective population size in interval [0, T1]
# N_e2 = 500  # Effective population size in interval [T1, T2]
# N_e3 = 2500   # Effective population size in interval [T2, infinity]
T1 = 300    # First time boundary in generations
T2 = 350
a_values = np.arange(0, n + 1)
b_values = np.arange(0,n + 1)
def safe_logsumexp(log_terms,signs=None):
    if len(log_terms) == 0:
        return -np.inf 
    max_x = np.max(log_terms) 
    scale=max_x
    if max_x<np.log(1.0e-40):
        scale=np.log(1.0e-40) 
    shifted_exps = np.exp(log_terms - scale)# Compute e^(-(x_i - min_x))
    if signs is not None:
        shifted_exps *= signs  # Apply sign adjustments if given
    inner_sum = np.sum(shifted_exps)  # Sum up scaled exponentials
    if inner_sum <= 0:
        return -np.inf  # Prevent log(0) issues
    log_inner_sum = np.log(inner_sum) + scale  # Compute final log sum exp
    return log_inner_sum

# Efficiently calculate rising factorial in log space
def log_rising_factorial(a, b):
    if b == 0 or a==0:
        return 0
    if a < b:
        return 0  
    return np.sum([np.log(a - i) for i in range(b)])

# Efficiently calculate falling factorial in log space
def log_falling_factorial(a, n):
    if n == 0 or a==0:
        return 0
    return np.sum([np.log(a + i) for i in range(n)])

# Generate dataframes for rising and falling factorials
def create_factorial_dataframes(a_values, b_values):
    rising_data = {
        f'a[{b}]': [log_rising_factorial(a, b) for a in a_values] for b in b_values
    }
    falling_data = {
        f'a_{n}': [log_falling_factorial(a, n) for a in a_values] for n in b_values
    }
    df_rising = pd.DataFrame(rising_data, index=a_values)
    df_falling = pd.DataFrame(falling_data, index=a_values)
    return df_rising, df_falling

# Function to calculate "choose 2 out of k" (k choose 2)
def choose_2_out_of_k(k):
    if k < 2:
        return 0
    return k*(k-1)/2
C1 = np.array([choose_2_out_of_k(k) for k in range(n + 1)])


df_rising, df_falling = create_factorial_dataframes(a_values, b_values)  


def logconstant_tables_Anp(n,Ne=1,t=1):
    signs_arr = np.empty((n + 1, n + 1), dtype=object)
    log_factorial=np.empty((n + 1, n + 1), dtype=object)
    exp_arr=np.empty((n + 1, n + 1), dtype=object)
    bk=np.empty(n )
    ck=np.empty(n )
    for k_start in range(1,n+1):
        for k_end in range(1,k_start+1):
            log_terms= []
            signs=[]
            exp_term=[]
            bk[k_end-1]=np.exp(df_rising.iloc[n, k_end]-df_falling.iloc[n, k_end]+np.log(2*k_end-1))
            ck[k_end-1]=-C1[k_end]*t/2
            for k in range(k_end, k_start + 1):
                # Numerator and denominator in log space
                numerator = (
                    np.log(2 * k - 1)
                    + df_falling.iloc[k_end, k - 1]
                    + df_rising.iloc[k_start, k]
                )

                denominator = (
                gammaln(k_end + 1)  # log(k_end!)
                + gammaln(k - k_end + 1)  # log((k - k_end)!)
                + df_falling.iloc[k_start, k]
                )
                signs.append(1-2*((k - k_end)%2))
                log_terms.append(numerator - denominator)
                exp_term.append(-C1[k]*t/2/Ne)

            signs_arr[k_start, k_end] = np.array(signs)
            log_factorial[k_start, k_end] = np.array(log_terms)
            exp_arr[k_start, k_end] = np.array(exp_term)
    return signs_arr, log_factorial,exp_arr,bk,ck

signs_arr, log_factorial,exp_arr,bk,ck = logconstant_tables_Anp(n)

def Ant_p(Ne,k_start, k_end,t):
    log_terms=exp_arr[k_start,k_end]*t/Ne+log_factorial[k_start,k_end]
    signs=signs_arr[k_start,k_end]
    # Combine positive and negative parts safely
    log_inner_sum = safe_logsumexp(log_terms,signs=signs)
    return log_inner_sum

def log_explicit_integral(mu, sigma, x1, x2, C):
    mu_new = mu - C * sigma**2  # Adjusted mean
    sigma_new = sigma           # Standard deviation

    z1 = (x1-mu_new)/sigma_new
    z2 = (x2-mu_new)/sigma_new

    # Compute log CDF values
    if T2 == np.inf:
        log_cdf_value = -0.717*z1 - 0.416*z1**2   # Approximation for upper tail
    else:

        log_cdf_value = -0.717*z1 - 0.416*z1**2 + 0.717*z2 + 0.416*z2**2  # Approximation for lower tail

    # Compute log constant term
    log_constant = (C * (x1-mu) + 0.5 * C**2 * sigma**2) - norm.logsf(0, loc=mu, scale=sigma)
    log_result = log_cdf_value + log_constant
    return log_result


def log_Postk_in_interval(mu,sigma,T1,T2,kth,n,Ne):  
    log_terms= []
    signs=[]
    k_end=n-kth+1
    for k in range(k_end,n+1):
        numerator = (
            np.log(2 * k - 1)
            + df_falling.iloc[k_end, k - 1]
            + df_rising.iloc[n, k]
        )

        denominator = (
        gammaln(k_end + 1)  # log(k_end!)
        + gammaln(k - k_end + 1)  # log((k - k_end)!)
        + df_falling.iloc[n, k]
        )
        intergral=log_explicit_integral(mu,sigma,T1,T2,C1[k]/(2*Ne))   #here attention, 2*Ne
        log_terms.append(intergral + numerator - denominator)
        # print(k)
        # print(np.exp(intergral))
        # print(np.exp(numerator - denominator))
        signs.append(1-2*((k - (k_end))%2))
        
    log_inner_sum = safe_logsumexp(log_terms,signs=signs)+ np.log((n-kth+1)*(n-kth)/4/Ne)
    return log_inner_sum




def dynamic_programming(n, N_e1, N_e2, N_e3, T1, T2):
    t = np.array([T1, T2 - T1])
    num_interval = len(t) + 1
    Ne = np.array([N_e1, N_e2, N_e3])
    max_sum = n - 1

    # Initialize log_dp table: all -inf, except base case
    log_dp = np.full((max_sum + 1, num_interval), -np.inf)
    log_dp[0][0] = 0.0  # log(1) = 0

    # Fill log_dp table
    for i in range(1, num_interval):
        for s in range(max_sum + 1):
            terms = []
            for j in range(s + 1):
                log_prev = log_dp[s - j][i - 1]
                if np.isneginf(log_prev):
                    continue
                ant_val = Ant_p(Ne[i - 1], n - (s - j), n - s, t[i - 1])
                log_term = log_prev + ant_val
                terms.append(log_term)
            if terms:
                log_dp[s][i] = safe_logsumexp(np.array(terms))

    return log_dp

In [2]:
#here the i can be really tricky, it is the index of the mu_vec and sigma_vec,
#so if we want to let i represent the index of coalenscence, we should always have i+1, for i could be 0
#second error is forgetting to set the new starting lineags n-l
def log_posexp_1(mu_vec,sigma_vec,T1,n,N_e1):
    log_Post_expectation=[]
    for i in range(1,n):#i+1=1,2.....19
        p_temp=log_Postk_in_interval(mu_vec[i-1],sigma_vec[i-1],0,T1,i,n,N_e1)
        # if p_temp==-np.inf:
        #     print(f'log_Postk_in_interval is -inf, i={i}')
        #     print(mu_vec[i-1])
        #     print(sigma_vec[i-1])
        log_Post_expectation.append(p_temp)
    return log_Post_expectation
def log_posexp_2(mu_vec,sigma_vec,T1,T2,n,N_e2,dp):
    log_Post_expectation=[]
    for i in range(1,n):#i+1=1,2.....19
        log_P_total_i=[]
        for s in range(i):#s=0,1,2,....18
            prob_B_given_A = log_Postk_in_interval(mu_vec[i-1],sigma_vec[i-1],T1,T2,i-s,n-s,N_e2)
            # if prob_B_given_A==-np.inf:
            #     print(f'log_Postk_in_interval is -inf, i={i},s={s}')
            prob_A_s = np.log(dp[s][1])                                    #i+1 here should also minus s>
            log_P_total_i.append(prob_A_s+prob_B_given_A)
        log_Post_expectation.append(log_P_total_i)
    return log_Post_expectation
def log_posexp_3(mu_vec,sigma_vec,T2,n,N_e3,dp,T3=np.inf):
    log_Post_expectation=[]
    for i in range(1,n):
        log_P_total_i=[]
        for s in range(i):
            prob_A_s = np.log(dp[s][2])
            prob_B_given_A = log_Postk_in_interval(mu_vec[i-1],sigma_vec[i-1],T2,T3,i-s,n-s,N_e3)
            # if prob_B_given_A==-np.inf:
            #     print(f'log_Postk_in_interval is -inf, i={i},s={s}')
            log_P_total_i.append(prob_A_s+prob_B_given_A)
        log_Post_expectation.append(log_P_total_i)
    return log_Post_expectation

def normalize_log_posterior(mu_vec,sigma_vec,T1,T2,n,N_e1,N_e2,N_e3):
    dp=dynamic_programming(n,N_e1,N_e2,N_e3,T1,T2)
    log_posterior1,log_posterior2,log_posterior3=log_posexp_1(mu_vec,sigma_vec,T1,n,N_e1),log_posexp_2(mu_vec,sigma_vec,T1,T2,n,N_e2,dp),log_posexp_3(mu_vec,sigma_vec,T2,n,N_e3,dp)
    for i in range(len(log_posterior1)):
        constant=safe_logsumexp(np.array([log_posterior1[i]]+log_posterior2[i]+log_posterior3[i]))
        log_posterior1[i]=log_posterior1[i]-constant
        log_posterior2[i]=safe_logsumexp(log_posterior2[i])-constant
        log_posterior3[i]=safe_logsumexp(log_posterior3[i])-constant
    col1_pos=np.exp(safe_logsumexp(log_posterior1))
    col2_pos=np.exp(safe_logsumexp(log_posterior2))
    col3_pos=np.exp(safe_logsumexp(log_posterior3))
    return col1_pos,col2_pos,col3_pos

In [3]:
invphi = (math.sqrt(5) - 1) / 2  # 1 / phi


def gss(f, a, b, arg1,arg2,arg3,tolerance=1e-5):
    """
    Golden-section search
    to find the minimum of f on [a,b]

    * f: a strictly unimodal function on [a,b]

    Example:
    >>> def f(x): return (x - 2) ** 2
    >>> x = gss(f, 1, 5)
    >>> print(f"{x:.5f}")
    2.00000

    """
    while b - a > tolerance:
        c = b - (b - a) * invphi
        d = a + (b - a) * invphi
        fc=f(c,arg1,arg2,arg3)    
        fd=f(d,arg1,arg2,arg3)
        if fc < fd:
            b = d
        else:  # f(c) > f(d) to find the maximum
            a = c

    return (b + a) / 2

In [7]:
#input data, run only once
import tskit
file_path = "/space/s1/KaiyuanLi/Ne_estimate/Non_use/ratio1/1/arg50_1.trees"
ts = tskit.load(file_path)
# Path to the .trees file
num_tree=ts.num_trees
num_arg=100
span=[]
# Load the tree sequence
mu_vec=np.zeros([num_tree,n-1])
sigma_vec=np.zeros([num_tree,n-1])
time=np.zeros([n-1,ts.num_trees,num_arg])
n2count=0
for j in range(num_arg):
    file_path = f"/space/s1/KaiyuanLi/Ne_estimate/Non_use/ratio1/1/arg50_{j+1}.trees"
    ts = tskit.load(file_path)
    # Iterate over each tree and find the 19th coalescence time
    for tree in ts.trees():
        coalescence_times = [
        ts.node(node).time for node in tree.nodes()
        if  ts.node(node).time > 0]
        # Sort times in ascending order
        coalescence_times.sort()
        if j==1:
            span.append(int(tree.interval[1] - tree.interval[0]))
        # print(tree.index)
        # Get the 19th coalescence time (if enough events exist)
        if len(coalescence_times) == n-1:
            for i in range(n-1):
                time[i,tree.index,j]=coalescence_times[i]
                if T1 <= coalescence_times[i]<T2:
                    n2count+=1
                

# label the node and caluclate the node specific 
# some labeling of the arg.

In [ ]:
#time=np.zeros([n-1,ts.num_trees,num_arg])
# using mean of exponiential
from scipy.stats import norm,truncnorm
from scipy.optimize import minimize

mu_vec_nt=np.zeros([num_tree,n-1])
sigma_vec_nt=np.zeros([num_tree,n-1])
def truncated_normal_mle(x):
    """
    Compute MLE estimates for a truncated normal distribution at 0.
    
    Parameters:
        x (array-like): Observed data (assumed to be truncated at 0)
    
    Returns:
        mu_hat, sigma_hat: Estimated parameters
    """
    x = np.array(x)
    n = len(x)
    
    # Initial estimates using method of moments
    mu_init = np.mean(x)
    sigma_init = np.std(x, ddof=1)

    def neg_log_likelihood(params):
        mu, sigma = params
        if sigma <= 0:  # Ensure positive sigma
            return np.inf
        z = -mu / sigma
        trunc_factor = 1 - norm.cdf(z)
        log_likelihood = (
            -n * np.log(sigma) 
            - np.sum((x - mu) ** 2) / (2 * sigma ** 2)
            - n * np.log(trunc_factor)
        )
        return -log_likelihood  # Negative for minimization

    # Optimize using scipy's minimize function
    result = minimize(
        neg_log_likelihood, 
        x0=[mu_init, sigma_init], 
        bounds=[(None, None), (1e-6, None)],  # Ensure sigma > 0
        method="L-BFGS-B"
    )
    
    mu_hat, sigma_hat = result.x
    return mu_hat, sigma_hat

# Generate synthetic data from a normal distribution truncated at 0
for i in range(n-1):
    for j in range(num_tree):
        mu_vec[j,i],sigma_vec[j,i]=truncated_normal_mle(time[i,j,:])
        # print(mu_vec[j,i],sigma_vec[j,i])



/tmp/ipykernel_3225610/11690210.py:34: RuntimeWarning: divide by zero encountered in log
  - n * np.log(trunc_factor)
/home/KaiyuanLi/miniconda3/lib/python3.11/site-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]


In [ ]:
from scipy.optimize import root_scalar
def solve_for_x(C, b, c, x_left=300, x_right=3000):
    def f(x):
        return np.sum(b * np.exp(c / x)) - C

    sol = root_scalar(f, bracket=[x_left, x_right], method='brentq')
    if sol.converged:
        return sol.root
    else:
        raise RuntimeError("Root finding did not converge.")

In [ ]:
import random
ran_num = 100
Ne_1,Ne_2,Ne_3=1000,1000,1000
seed=0
random.seed(seed)
selected_indices = random.sample(range(ts.num_trees), ran_num) 
subspan=np.array(span)[selected_indices]
subspan_rate=subspan/np.sum(subspan)
while True:
    col1=[]
    col2=[]
    col3=[]
    Ne1,Ne2,Ne3=Ne_1,Ne_2,Ne_3
    # random.seed(seed)
    # selected_indices = random.sample(range(ts.num_trees), ran_num)  # Randomly select ran_num indices
    for i in selected_indices:
        col1_pos,col2_pos,col3_pos=normalize_log_posterior(mu_vec[i,:],sigma_vec[i,:],T1,T2,n,Ne1,Ne2,Ne3)
        col1.append(col1_pos)
        col2.append(col2_pos)
        col3.append(col3_pos)
    # subspan=np.array(span)[selected_indices]
    # subspan_rate=subspan/np.sum(subspan)
    mean_ni1=np.average(np.array(col1),weights=subspan_rate)
    Ne_1=solve_for_x(n-mean_ni1,bk,ck*T1)
    Ne_2=500
    Ne_3=2500
    print(Ne_1,Ne_2,Ne_3)
    seed+=1
    if np.abs(Ne1-Ne_1)+np.abs(Ne2-Ne_2)+np.abs(Ne3-Ne_3)<1: #and np.abs(Ne3-Ne_3)<1? might not be important anyway
        break

In [ ]:
# use e^cx*phi(x) approximation, test sum is 1.
# compare e^cx*phi(x) approximation with explicit integral